In [1]:
!pip install torch transformers rouge-score accelerate datasets



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\ashwi\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch

from transformers import pipeline
from rouge_score import rouge_scorer

c:\Users\ashwi\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("D:\\news\\Classfication\\news.tsv", sep="\t")

In [4]:
df = df.dropna(subset=["Headline", "News body", "Category"])

df["article"] = df["Headline"].astype(str) + " " + df["News body"].astype(str)
df["summary"] = df["Headline"].astype(str)

# Use SAME 50% sampling as notebook 1
df = df.sample(frac=0.5, random_state=42).reset_index(drop=True)

df.head()

,News ID,Category,Topic,Headline,News body,Title entity,Entity content,article,summary
0,N95333,news,newsus,This dog's smile will melt your heart,"Mocca lives in Yokohama, Japan, and is a Shiba...",{},{},This dog's smile will melt your heart Mocca li...,This dog's smile will melt your heart
1,N41910,lifestyle,shop-all,The Most Popular Walmart Item in Every State,What are the most oft-ordered Walmart products...,{'Walmart': 'Walmart'},"{'Walmart': {'type': 'item', 'id': 'Q18615334'...",The Most Popular Walmart Item in Every State W...,The Most Popular Walmart Item in Every State
2,N88506,finance,finance-real-estate,Photos: Look Glenn Close's 'Beanfield' estate ...,Emmy winning actress Glen Close listed her Bed...,"{""Glenn Close's"": 'Glenn Close', 'Bedford': 'B...","{'Glenn Close': {'type': 'item', 'id': 'Q37231...",Photos: Look Glenn Close's 'Beanfield' estate ...,Photos: Look Glenn Close's 'Beanfield' estate ...
3,N114168,news,newscrime,Hillsborough Sheriff's Office sweep results in...,TAMPA More than 80 people have been arrested...,{'human trafficking': 'Human trafficking'},"{'Human trafficking': {'type': 'item', 'id': '...",Hillsborough Sheriff's Office sweep results in...,Hillsborough Sheriff's Office sweep results in...
4,N35279,video,peopleandplaces,Family of missing Connecticut mom blast 'Gone ...,Family members and friends of Jennifer Dulos s...,{'Connecticut': 'Connecticut'},"{'Connecticut': {'type': 'item', 'id': 'Q58425...",Family of missing Connecticut mom blast 'Gone ...,Family of missing Connecticut mom blast 'Gone ...


In [5]:
baseline_df = pd.read_csv("D:\\news\\Summarization\\from_scratch_summarization_results.csv")
baseline_df

,Model,ROUGE-1,ROUGE-2,ROUGE-L,Average
0,TF-IDF,0.159553,0.148539,0.159553,0.155882
1,TextRank,0.159553,0.148539,0.159553,0.155882
2,Seq2Seq,0.107275,0.042570,0.096983,0.082276


In [6]:
from transformers import (
    BertTokenizer, BertModel,
    T5Tokenizer, T5ForConditionalGeneration
)

In [7]:
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")
bert_model.eval()

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 140.68it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [8]:
def bert_sentence_embedding(sentence):
    inputs = bert_tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )
    with torch.no_grad():
        outputs = bert_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

In [9]:
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from nltk.tokenize import sent_tokenize

In [10]:
def bert_extractive_summarize(text, top_n=3):
    sentences = sent_tokenize(text)

    if len(sentences) <= top_n:
        return " ".join(sentences)

    embeddings = np.array(
        [bert_sentence_embedding(s) for s in sentences]
    )

    doc_embedding = embeddings.mean(axis=0).reshape(1, -1)
    scores = cosine_similarity(embeddings, doc_embedding).flatten()

    ranked = scores.argsort()[::-1][:top_n]
    ranked = sorted(ranked)

    return " ".join([sentences[i] for i in ranked])

In [11]:
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")
t5_model.eval()

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 159.72it/s, Materializing param=shared.weight]                                                      


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [12]:
def t5_summarize(text, max_len=120):
    input_text = "summarize: " + text[:512]

    inputs = t5_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():
        summary_ids = t5_model.generate(
            inputs["input_ids"],
            max_length=max_len,
            min_length=40,
            num_beams=2,
            early_stopping=True
        )

    return t5_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

In [13]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def evaluate_model(fn, df, samples=30):
    r1, r2, rL = [], [], []

    for i in range(samples):
        ref = df.iloc[i]["summary"]
        pred = fn(df.iloc[i]["article"])

        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rL.append(scores["rougeL"].fmeasure)

    return np.mean(r1), np.mean(r2), np.mean(rL)

In [14]:
bert_scores = evaluate_model(bert_extractive_summarize, df)
t5_scores = evaluate_model(t5_summarize, df)

results = pd.DataFrame([
    {
        "Model": "BERT-base (Extractive)",
        "ROUGE-1": bert_scores[0],
        "ROUGE-2": bert_scores[1],
        "ROUGE-L": bert_scores[2]
    },
    {
        "Model": "T5-small (Abstractive)",
        "ROUGE-1": t5_scores[0],
        "ROUGE-2": t5_scores[1],
        "ROUGE-L": t5_scores[2]
    }
])

results["Average"] = results[["ROUGE-1", "ROUGE-2", "ROUGE-L"]].mean(axis=1)
results.round(4)

,Model,ROUGE-1,ROUGE-2,ROUGE-L,Average
0,BERT-base (Extractive),0.187,0.1233,0.1714,0.1606
1,T5-small (Abstractive),0.335,0.2508,0.3174,0.3011


REFERENCE

In [15]:
best_model = results.sort_values("Average", ascending=False).iloc[0]

best_config = {
    "model": best_model["Model"],
    "rouge_avg": best_model["Average"]
}

import joblib
joblib.dump(best_config, "best_transformer_summarizer.pkl")

['best_transformer_summarizer.pkl']

In [31]:
MODELS = {
    "T5-small": "t5-small",
    "T5-base": "t5-base",
    "BART-base": "facebook/bart-base"
}

In [41]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def evaluate_transformer(model_name, df, samples=30):
    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
    summarizer = lambda text: tokenizer.decode(
        model.generate(tokenizer(text, return_tensors="pt", truncation=True, max_length=512).input_ids, 
                      max_new_tokens=80)[0], skip_special_tokens=True
    )
    r1, r2, rL = [], [], []

    for i in range(samples):
        text = df.iloc[i]["article"][:1500]  # truncate for safety
        ref = df.iloc[i]["summary"]

        pred = summarizer(text)[:80]

        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rL.append(scores["rougeL"].fmeasure)

    return {
        "rouge1": np.mean(r1),
        "rouge2": np.mean(r2),
        "rougeL": np.mean(rL)
    }

In [42]:

transformer_results = []

for name, model_id in MODELS.items():
    print(f"Evaluating {name}...")
    scores = evaluate_transformer(model_id, df)

    transformer_results.append({
        "Model": name,
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"]
    })

transformer_df = pd.DataFrame(transformer_results)
transformer_df["Average"] = transformer_df[
    ["rouge1", "rouge2", "rougeL"]
].mean(axis=1)

transformer_df.round(4)

Evaluating T5-small...


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 222.67it/s, Materializing param=shared.weight]                                                      


Evaluating T5-base...


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 401.35it/s, Materializing param=shared.weight]                                                      


Evaluating BART-base...


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 393.01it/s, Materializing param=shared.weight]                                                      


,Model,rouge1,rouge2,rougeL,Average
0,T5-small,0.3433,0.263,0.3116,0.3059
1,T5-base,0.3433,0.263,0.3116,0.3059
2,BART-base,0.3433,0.263,0.3116,0.3059


In [43]:
combined_df = pd.concat(
    [baseline_df, transformer_df],
    ignore_index=True
)

combined_df = combined_df.round(4)
combined_df

,Model,ROUGE-1,ROUGE-2,ROUGE-L,Average,rouge1,rouge2,rougeL
0,TF-IDF,0.1596,0.1485,0.1596,0.1559,NaN,NaN,NaN
1,TextRank,0.1596,0.1485,0.1596,0.1559,NaN,NaN,NaN
2,Seq2Seq,0.1073,0.0426,0.0970,0.0823,NaN,NaN,NaN
3,T5-small,NaN,NaN,NaN,0.3059,0.3433,0.263,0.3116
4,T5-base,NaN,NaN,NaN,0.3059,0.3433,0.263,0.3116
5,BART-base,NaN,NaN,NaN,0.3059,0.3433,0.263,0.3116


In [44]:
combined_df.to_csv(
    "final_summarization_comparison.csv",
    index=False
)

print("Saved → final_summarization_comparison.csv")

Saved → final_summarization_comparison.csv


In [45]:
best_row = combined_df.sort_values(
    "Average",
    ascending=False
).iloc[0]

best_model = {
    "model_name": best_row["Model"],
    "rouge1": best_row["rouge1"],
    "rouge2": best_row["rouge2"],
    "rougeL": best_row["rougeL"],
    "average": best_row["Average"],
    "model_type": "Transformer"
}

import joblib
joblib.dump(best_model, "best_transformer_summarizer.pkl")

best_model

{'model_name': 'T5-small',
 'rouge1': 0.3433,
 'rouge2': 0.263,
 'rougeL': 0.3116,
 'average': 0.3059,
 'model_type': 'Transformer'}